# 02 - Carga a Supabase/PostgreSQL
Objetivo: cargar CSV por lotes a las tablas finales.

In [ ]:
import os
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine
from dotenv import load_dotenv
load_dotenv()
engine = create_engine(os.environ['SUPABASE_DB_URL'])
DATA_DIR = Path(os.getenv('DATA_DIR', '/content/data'))

In [ ]:
# Ejemplo de carga. Ajustar nombres si los CSV tienen sufijo.
load_order = [
    ('customers', 'olist_customers_dataset.csv'),
    ('geolocations', 'olist_geolocation_dataset.csv'),
    ('sellers', 'olist_sellers_dataset.csv'),
    ('product_categories', 'product_category_name_translation.csv'),
    ('products', 'olist_products_dataset.csv'),
    ('orders', 'olist_orders_dataset.csv'),
    ('order_payments', 'olist_order_payments_dataset.csv'),
    ('order_reviews', 'olist_order_reviews_dataset.csv'),
]
for table, file in load_order:
    df = pd.read_csv(DATA_DIR / file)
    df.to_sql(table, engine, if_exists='append', index=False, chunksize=5000, method='multi')
    print('loaded', table, len(df))

In [ ]:
with engine.begin() as conn:
    conn.exec_driver_sql("""
    INSERT INTO orders_part
    SELECT order_id, customer_id, order_status, order_purchase_timestamp, order_approved_at,
           order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date, raw_payload
    FROM orders
    ON CONFLICT DO NOTHING;
    ANALYZE;
    """)